# Figure Data Extraction — Tooling Test Notebook

This notebook evaluates open-source tools for pulling **data series out of research-paper figures**,
so digitized curves (e.g. microbial count vs. day, or an FTIR spectrum) can feed the long-format
pipeline in [`../notebooks/data_processing.ipynb`](../notebooks/data_processing.ipynb).

It has two parts:

- **Part 1 — Image extraction from PDFs** (PyMuPDF / `fitz`). Pulls every embedded raster image out
  of a PDF and groups them into a per-PDF subfolder. This part already works; here it is wrapped in a
  reusable function.
- **Part 2 — Chart-to-table extraction** ([PP-Chart2Table](https://huggingface.co/PaddlePaddle/PP-Chart2Table)).
  A vision-language model that reads a chart image directly and returns a data table — no manual axis
  calibration, no per-chart-type tool choice, and no panel cropping required. Works on line, bar, and
  scatter charts alike, and on **any image path** you point it at.

> LineFormer (line-charts only, needs manual panel cropping) and plotdigitizer (needs hand-supplied
> pixel↔value calibration per chart) were tried first but required per-chart manual setup that doesn't
> generalize. PP-Chart2Table replaces both: point it at an image path and it returns a table directly.


## Part 0 — Setup

Dependencies:

- **`fitz` (PyMuPDF)** — open PDFs and extract embedded images (Part 1).
- **`paddleocr`** (>=3.5) + **`paddlepaddle`** — runs `PP-Chart2Table` via the `ChartParsing` module (Part 2).
- **`pandas`** — parse the model's markdown-table output into a DataFrame.


In [1]:
import os
from pathlib import Path
from io import StringIO

import fitz  # PyMuPDF
import pandas as pd
from paddleocr import ChartParsing


C:\Users\safae\projects\ML\misc\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1 — Image extraction from PDFs (PyMuPDF)

`extract_images()` walks every page of a PDF, pulls out each embedded raster image, and writes them to
`<output_root>/<pdf_stem>/page{N}_img{M}.{ext}`. Grouping by PDF stem keeps images from different papers
(e.g. `11.pdf` vs `12.pdf`) in separate folders instead of colliding in one flat directory.


In [2]:
def extract_images(pdf_path, output_root="extracted_images"):
    """Extract all embedded raster images from a PDF into a per-PDF subfolder.

    Images are written to ``<output_root>/<pdf_stem>/page{N}_img{M}.{ext}``.
    Returns the list of written file paths.
    """
    pdf_path = Path(pdf_path)
    out_dir = Path(output_root) / pdf_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    written = []
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            image_list = doc[page_num].get_images(full=True)
            for img_index, img_info in enumerate(image_list):
                xref = img_info[0]
                base_image = doc.extract_image(xref)
                ext = base_image["ext"]
                img_path = out_dir / f"page{page_num + 1}_img{img_index + 1}.{ext}"
                img_path.write_bytes(base_image["image"])
                written.append(img_path)

    print(f"Extracted {len(written)} images from {pdf_path.name} -> {out_dir}/")
    return written


In [3]:
# Demonstrate on the two sample papers in this folder.
paths_11 = extract_images("11.pdf")
paths_12 = extract_images("12.pdf")


Extracted 11 images from 11.pdf -> extracted_images\11/
Extracted 12 images from 12.pdf -> extracted_images\12/


## Part 2 — Chart-to-table extraction (PP-Chart2Table)

`chart_to_df()` is **plot-agnostic**: pass it any chart image path and it returns a `DataFrame` of the
digitized data, no axis calibration or panel cropping needed. Swap `CHART_IMG` below for any image in
`extracted_images/` and re-run.


In [4]:
# Loaded once; reused across images. The "transformers" engine avoids a packaging bug where
# paddlex's native CPU backend unconditionally imports GPU-only fused kernels for an unrelated
# sibling model (see https://github.com/PaddlePaddle/PaddleX/issues/4704).
_chart_model = ChartParsing(
    model_name="PP-Chart2Table",
    engine="transformers",
    engine_config={"dtype": "float32"},
    device="cpu",
)


def chart_to_df(image_path):
    """Run PP-Chart2Table on any chart image and return its extracted table as a DataFrame."""
    image_path = Path(image_path)
    results = _chart_model.predict(input={"image": str(image_path)}, batch_size=1)
    result = next(iter(results))
    markdown_table = result["result"]

    df = pd.read_csv(StringIO(markdown_table), sep="|", engine="python")
    df = df.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(axis=1, how="all")  # drop stray empty columns from leading/trailing pipes
    return df.reset_index(drop=True)

Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\safae\.paddlex\official_models\PP-Chart2Table_safetensors`.


[transformers] The tokenizer you are loading from 'C:\Users\safae\.paddlex\official_models\PP-Chart2Table_safetensors' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 471/471 [00:00<00:00, 10144.17it/s]

### Run on a chart image

Only the path changes between charts — no other setup.


In [5]:
CHART_IMG = Path("extracted_images/12/page8_img1.jpeg")

chart_df = chart_to_df(CHART_IMG)
print(f"Extracted {len(chart_df)} rows, columns: {list(chart_df.columns)}")
chart_df

Extracted 5 rows, columns: ['Storage time (day)', 'Control1', 'Control2', 'Teo2%', 'Teo1%', 'Se02%', 'Se01%', 'Te0+Se02%', 'Te0+Se01%']


,Storage time (day),Control1,Control2,Teo2%,Teo1%,Se02%,Se01%,Te0+Se02%,Te0+Se01%
0,1,4.6,2.2,3.2,3.8,4.4,3.5,3.3,4.1
1,4,5.7,3.1,3.8,4.8,4.2,4.5,3.4,3.9
2,8,6.7,3.4,4.8,5.4,5.9,5.5,4.9,5.6
3,10,6.9,3.5,4.9,6.4,6.7,6.5,4.8,6.5
4,12,7.8,3.9,4.8,6.8,6.9,6.6,4.9,5.1


## Part 3 — Image extraction comparison: docling vs PyMuPDF

PyMuPDF pulls **embedded raster XObjects**; docling pulls **detected picture regions** (rendered at
`IMAGE_RESOLUTION_SCALE`), so a vector figure or a multi-part panel can appear as one docling picture
but zero or many PyMuPDF images. Set `PDF_PATH` below.

In [ ]:
import time

import matplotlib.pyplot as plt
from PIL import Image

from docling_core.types.doc import PictureItem
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

PDF_PATH = Path("path/to/your/paper.pdf")  # <-- your input
COMPARE_ROOT = Path("comparison_images")
IMAGE_RESOLUTION_SCALE = 2.0

In [ ]:
def extract_pymupdf(pdf_path, output_root=COMPARE_ROOT):
    pdf_path = Path(pdf_path)
    out_dir = Path(output_root) / pdf_path.stem / "pymupdf"
    out_dir.mkdir(parents=True, exist_ok=True)

    written, start = [], time.perf_counter()
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            for img_index, img_info in enumerate(doc[page_num].get_images(full=True)):
                base_image = doc.extract_image(img_info[0])
                img_path = out_dir / f"page{page_num + 1}_img{img_index + 1}.{base_image['ext']}"
                img_path.write_bytes(base_image["image"])
                written.append(img_path)
    return {"tool": "pymupdf", "paths": written, "seconds": time.perf_counter() - start}


def extract_docling(pdf_path, output_root=COMPARE_ROOT, scale=IMAGE_RESOLUTION_SCALE):
    pdf_path = Path(pdf_path)
    out_dir = Path(output_root) / pdf_path.stem / "docling"
    out_dir.mkdir(parents=True, exist_ok=True)

    pipeline_options = PdfPipelineOptions()
    pipeline_options.images_scale = scale
    pipeline_options.generate_page_images = True
    pipeline_options.generate_picture_images = True
    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
    )

    written, start = [], time.perf_counter()
    conv_res = converter.convert(pdf_path)
    for element, _level in conv_res.document.iterate_items():
        if isinstance(element, PictureItem):
            image = element.get_image(conv_res.document)
            if image is None:
                continue
            img_path = out_dir / f"picture{len(written) + 1}.png"
            with img_path.open("wb") as fp:
                image.save(fp, "PNG")
            written.append(img_path)
    return {"tool": "docling", "paths": written, "seconds": time.perf_counter() - start}

In [ ]:
runs = [extract_pymupdf(PDF_PATH), extract_docling(PDF_PATH)]

summary = pd.DataFrame(
    [
        {
            "tool": r["tool"],
            "n_images": len(r["paths"]),
            "seconds": round(r["seconds"], 2),
            "megapixels": round(
                sum(Image.open(p).width * Image.open(p).height for p in r["paths"]) / 1e6, 2
            ),
        }
        for r in runs
    ]
)
summary

In [ ]:
n_cols = max(len(r["paths"]) for r in runs) or 1
fig, axes = plt.subplots(
    len(runs), n_cols, figsize=(2.4 * n_cols, 3.0 * len(runs)), squeeze=False
)

for row, run in enumerate(runs):
    for col in range(n_cols):
        ax = axes[row][col]
        ax.axis("off")
        if col < len(run["paths"]):
            path = run["paths"][col]
            with Image.open(path) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_title(f"{path.name}\n{im.width}x{im.height}", fontsize=7)
    axes[row][0].set_ylabel(run["tool"])
    axes[row][0].axis("on")
    axes[row][0].set_xticks([])
    axes[row][0].set_yticks([])

fig.suptitle(f"{PDF_PATH.name} — " + " vs ".join(f"{r['tool']} ({len(r['paths'])})" for r in runs))
fig.tight_layout()
plt.show()